# ▶️ Running dbt + Snowflake Commands from Prefect Tasks

---

## 🤔 How It Works

Prefect uses Python's **`subprocess`** module to call the dbt CLI.  
dbt CLI reads `profiles.yml` → connects to **Snowflake** → runs models there.

```
Prefect @task → subprocess → dbt CLI → profiles.yml → Snowflake
```

---

## 💻 Example 1: Reusable dbt Runner (With Credential Injection)

In [ ]:
import subprocess
import os
from prefect import task, flow

DBT_PROJECT_DIR = "/Users/aviraljain/Downloads/python advanced/my_etl_project"

def inject_snowflake_env() -> dict:
    """
    Returns environment variables with Snowflake credentials.
    These override values in profiles.yml when env_var() is used.
    
    In production: load from Prefect Secrets:
        from prefect.blocks.system import Secret
        {'SNOWFLAKE_PASSWORD': Secret.load('snowflake-password').get()}
    """
    env = os.environ.copy()  # Start with current env
    env.update({
        "SNOWFLAKE_ACCOUNT":   os.getenv("SNOWFLAKE_ACCOUNT", "your_account"),
        "SNOWFLAKE_USER":      os.getenv("SNOWFLAKE_USER", "your_user"),
        "SNOWFLAKE_PASSWORD":  os.getenv("SNOWFLAKE_PASSWORD", "your_password"),
        "SNOWFLAKE_DATABASE":  os.getenv("SNOWFLAKE_DATABASE", "ANALYTICS"),
        "SNOWFLAKE_WAREHOUSE": os.getenv("SNOWFLAKE_WAREHOUSE", "COMPUTE_WH"),
    })
    return env


def run_dbt_command(command: str, extra_flags: list = None) -> str:
    """Run any dbt CLI command against Snowflake."""
    cmd = ["dbt"] + command.split()
    cmd += ["--project-dir", DBT_PROJECT_DIR, "--profiles-dir", DBT_PROJECT_DIR]
    if extra_flags:
        cmd += extra_flags

    print(f"▶ Running: {' '.join(cmd)}")
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        env=inject_snowflake_env()   # Inject Snowflake credentials
    )
    print(result.stdout[-600:] if result.stdout else "")
    if result.returncode != 0:
        raise RuntimeError(f"dbt failed:\n{result.stderr[-400:]}")
    return result.stdout

print("Helper defined ✅")

---

## 💻 Example 2: Individual dbt Prefect Tasks

In [ ]:
@task(name="dbt seed → Snowflake RAW", retries=1)
def dbt_seed():
    """Upload CSV seed files to Snowflake RAW schema."""
    run_dbt_command("seed")
    print("✅ Seeds loaded into Snowflake")


@task(name="dbt run → Snowflake models", retries=2, retry_delay_seconds=30)
def dbt_run(target: str = "dev"):
    """Run all dbt models on Snowflake. Use 'prod' target for production."""
    run_dbt_command(f"run --target {target}")
    print(f"✅ dbt models complete on Snowflake ({target})")


@task(name="dbt test → Snowflake", retries=0)
def dbt_test(target: str = "dev"):
    """Run data quality tests against Snowflake tables."""
    run_dbt_command(f"test --target {target}")
    print(f"✅ dbt tests passed on Snowflake ({target})")


@flow(name="Basic Snowflake dbt Flow", log_prints=True)
def basic_snowflake_dbt_flow(target: str = "dev"):
    dbt_seed()
    dbt_run(target=target)
    dbt_test(target=target)
    print("✅ Snowflake dbt pipeline complete!")

basic_snowflake_dbt_flow(target="dev")

---

## 💻 Example 3: Snowflake-Specific dbt Select Patterns

In [ ]:
@task(name="dbt run - staging only")
def dbt_run_staging(target: str = "dev"):
    """Only run staging models — creates views in Snowflake STAGING schema."""
    run_dbt_command(f"run --select staging.* --target {target}")


@task(name="dbt run - marts only")
def dbt_run_marts(target: str = "dev"):
    """Only run mart models — creates tables in Snowflake ANALYTICS schema."""
    run_dbt_command(f"run --select marts.* --target {target}")


@task(name="dbt run - full refresh")
def dbt_run_full_refresh(target: str = "dev"):
    """Force rebuild of all incremental tables in Snowflake."""
    run_dbt_command(f"run --target {target}", extra_flags=["--full-refresh"])


# Common Snowflake/dbt patterns:
print("Common Snowflake-specific dbt patterns:")
patterns = {
    "--target prod":          "Use prod Snowflake schema instead of dev",
    "--full-refresh":         "Rebuild incremental Snowflake tables from scratch",
    "--select tag:snowflake": "Only run models tagged 'snowflake'",
    "--select staging.*":     "Only run staging layer → Snowflake STAGING schema",
    "--select marts.*":       "Only run marts layer → Snowflake ANALYTICS schema",
}
for flag, desc in patterns.items():
    print(f"  dbt run {flag:<30} → {desc}")

---

## 💻 Example 4: Snowflake profiles.yml with env_var() for Security

In [ ]:
# Best practice profiles.yml using env_var() — never hardcode credentials!
secure_profiles_yml = """
# profiles.yml — Snowflake (secure, using env_var)
my_etl_project:
  target: dev
  outputs:
    dev:
      type: snowflake
      account:   \"{{ env_var('SNOWFLAKE_ACCOUNT') }}\"
      user:      \"{{ env_var('SNOWFLAKE_USER') }}\"
      password:  \"{{ env_var('SNOWFLAKE_PASSWORD') }}\"
      role:      TRANSFORMER
      warehouse: \"{{ env_var('SNOWFLAKE_WAREHOUSE', 'COMPUTE_WH') }}\"
      database:  \"{{ env_var('SNOWFLAKE_DATABASE', 'ANALYTICS') }}\"
      schema: dbt_dev
      threads: 4

    prod:
      type: snowflake
      account:   \"{{ env_var('SNOWFLAKE_ACCOUNT') }}\"
      user:      \"{{ env_var('SNOWFLAKE_USER') }}\"
      password:  \"{{ env_var('SNOWFLAKE_PASSWORD') }}\"
      role:      TRANSFORMER
      warehouse: \"{{ env_var('SNOWFLAKE_WAREHOUSE', 'COMPUTE_WH') }}\"
      database:  \"{{ env_var('SNOWFLAKE_DATABASE', 'ANALYTICS') }}\"
      schema: dbt_prod
      threads: 8
"""
print(secure_profiles_yml)
print("✅ credentials injection flow: Prefect Secret → os.environ → profiles.yml env_var()")

---

## 🏭 Summary

| dbt Command | Snowflake Effect | Prefect Task |
|---|---|---|
| `dbt seed` | Loads CSVs into Snowflake RAW schema | `dbt_seed()` |
| `dbt run` | Creates views/tables in Snowflake | `dbt_run(target)` |
| `dbt run --full-refresh` | Rebuilds incremental tables | `dbt_run_full_refresh()` |
| `dbt test` | Validates Snowflake table data | `dbt_test(target)` |

---

## ⚠️ Common Beginners' Mistakes

In [ ]:
mistakes = [
    ("No --target flag",                 "Always pass target (dev/prod) — different Snowflake schemas"),
    ("Hardcoded password in profiles",   "Use env_var() + Prefect Secrets for credentials"),
    ("No --full-refresh on first run",   "First run of incremental models needs --full-refresh"),
    ("Wrong warehouse name",             "Check SHOW WAREHOUSES in Snowflake to verify exact name"),
]
for mistake, fix in mistakes:
    print(f"❌ {mistake}")
    print(f"✅ Fix: {fix}\n")